In [1]:
#Import#

from cellpose import models, core, io, plot
from cellpose import train
from cellpose import metrics
import pickle, numpy as np
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio.v2 as imageio
import os, re
import skimage.filters
from skimage.feature import peak_local_max
from skimage.measure import regionprops
from skimage import exposure, measure, filters, img_as_float
from skimage.measure import regionprops
from skimage.segmentation import watershed
from skimage.segmentation import mark_boundaries
from skimage.filters import sobel
from skimage.feature import blob_dog
from skimage import data, color, filters, segmentation, measure, morphology
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
from scipy.spatial import distance_matrix
from scipy.ndimage import generate_binary_structure, binary_dilation
from scipy.spatial import Delaunay
from scipy.stats import qmc
import pandas as pd
from glob import glob
import cv2
import re
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed

from nucleus_worker_old_construction import process_single_nucleus

import time

In [2]:
#Data input from the User

folder_path = r'Y:\Group Members\Valentin Aubry\01_Data\Test_Data_Andreas\data'  # The r in r'Example\Pathway' is very important to get the raw input


# Check if the folder is empty or even exists
if os.path.isdir(folder_path) == True:
    print("Folder Path exists")

    # Getting the list of directories
    dir = os.listdir(folder_path)
    
    # Checking if the list is empty or not
    if len(dir) == 0:
        print("Empty directory")
    else:
        print("Not empty directory")
else:
    print("Folder path not found")

Folder Path exists
Not empty directory


In [3]:
# Data Loading


# High Throughput Data Loading
Cy5_data = glob(os.path.join(folder_path, '*Cy5 SEM.tif'))
DAPI_data = glob(os.path.join(folder_path, '*DAPI SEM.tif'))
DAPI_mask_data = glob(os.path.join(folder_path, '*DAPI SEM_seg.npy'))
FITC_data = glob(os.path.join(folder_path, '*FITC SEM.tif'))
TRITC_data = glob(os.path.join(folder_path, '*TRITC SEM.tif'))


# Sorting the data in case it isn't yet
Cy5_data.sort()
DAPI_data.sort()
DAPI_mask_data.sort()
FITC_data.sort()
TRITC_data.sort()


# Displaying all the data that was loaded
for i in range (len(DAPI_data)):
    print(os.path.basename(Cy5_data[i]))
    print(os.path.basename(DAPI_data[i]))
    print(os.path.basename(DAPI_mask_data[i]))
    print(os.path.basename(FITC_data[i]))
    print(os.path.basename(TRITC_data[i]))





ATR1_24h--W00032--P00001--Z00000--T00000--Cy5 SEM.tif
ATR1_24h--W00032--P00001--Z00000--T00000--DAPI SEM.tif
ATR1_24h--W00032--P00001--Z00000--T00000--DAPI SEM_seg.npy
ATR1_24h--W00032--P00001--Z00000--T00000--FITC SEM.tif
ATR1_24h--W00032--P00001--Z00000--T00000--TRITC SEM.tif
ATR1_24h--W00032--P00002--Z00000--T00000--Cy5 SEM.tif
ATR1_24h--W00032--P00002--Z00000--T00000--DAPI SEM.tif
ATR1_24h--W00032--P00002--Z00000--T00000--DAPI SEM_seg.npy
ATR1_24h--W00032--P00002--Z00000--T00000--FITC SEM.tif
ATR1_24h--W00032--P00002--Z00000--T00000--TRITC SEM.tif
ATR1_24h--W00032--P00003--Z00000--T00000--Cy5 SEM.tif
ATR1_24h--W00032--P00003--Z00000--T00000--DAPI SEM.tif
ATR1_24h--W00032--P00003--Z00000--T00000--DAPI SEM_seg.npy
ATR1_24h--W00032--P00003--Z00000--T00000--FITC SEM.tif
ATR1_24h--W00032--P00003--Z00000--T00000--TRITC SEM.tif
ATR1_24h--W00032--P00004--Z00000--T00000--Cy5 SEM.tif
ATR1_24h--W00032--P00004--Z00000--T00000--DAPI SEM.tif
ATR1_24h--W00032--P00004--Z00000--T00000--DAPI SEM_seg

In [4]:
# Loading Functions

def save_dataframe_to_csv(df, folder_path, base_name, data_type):
    """
    Save a DataFrame to CSV with Excel-compatible formatting.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to save
    folder_path : str
        Directory where file will be saved
    base_name : str
        Base filename (without extension)
    data_type : str
        Type of data being saved (e.g., 'foci', 'nucleus', 'cytoplasm')
    """
    filename = f"{folder_path}/{base_name}--{data_type}_DF_SEM.csv"
    
    # Write sep=, line for Excel compatibility
    with open(filename, 'w') as f:
        f.write('sep=,\n')
    
    # Write data with header
    df.to_csv(filename, mode='a', header=True, index=False)


def farthest_point_sampling(points, n_samples):
    """
    Selects a subset of points that are maximally spread apart 
    (farthest-point or maximin sampling).

    Parameters
    ----------
    points : ndarray of shape (N, D)
        The full set of candidate points (e.g., Sobol samples inside a hull).
    n_samples : int
        The number of points to select.

    Returns
    -------
    selected_points : ndarray of shape (n_samples, D)
        The subset of points that are as far apart as possible.

    Algorithm
    ---------
    1. Pick one random starting point.
    2. Compute distances from all points to this selected point.
    3. Iteratively:
       - Choose the point with the largest distance to the nearest selected point.
       - Add it to the selected set.
       - Update distances so that each point stores its distance to the nearest selected point.
    4. Continue until n_samples points are selected.

    This greedy algorithm ensures each new point lies in the largest remaining empty region.
    It produces a very even spatial coverage, similar to Poisson-disk sampling.
    """

    n = len(points)
    if n_samples > n:
        raise ValueError("n_samples cannot exceed number of available points")

    # Picking a random starting point
    selected_idx = [np.random.randint(0, n)]

    # Computing distances from all points to the first selected point
    # Each element in 'dists' stores the distance to the *nearest selected* point so far
    dists = np.linalg.norm(points - points[selected_idx[0]], axis=1)

    # Iteratively adding points farthest from the selected set
    for _ in range(1, n_samples):
        # Find the index of the point that is farthest from all selected points
        next_idx = np.argmax(dists)
        selected_idx.append(next_idx)

        # Update distances: for every point, 
        # store the distance to the nearest selected point (so far)
        new_dists = np.linalg.norm(points - points[next_idx], axis=1)
        dists = np.minimum(dists, new_dists)

    # Returning the selected subset
    return points[selected_idx]


In [14]:
#Check if the script is the main program(instead of being an imported module), this stops the workers that are spawned below
#for the paralellization from re-running the whole script
if __name__ == '__main__':    
    
    
    #timer to measure how long it takes
    start_time = time.time()
    
    #-----------Delaunay + Sobel-------------#
    
    # load the convex hull(3d shape that contains all the valid parameter iterations) and the 
    # bounding box(max and min of all the parameters of the comvex hull) parameters to iterate through the parameters
    with open(r"Y:\Group Members\Valentin Aubry\01_Data\Parameters\Hull\valid_parameter_hull.pkl", "rb") as f:
        delaunay = pickle.load(f)
    # The bounding box is used to fill it with random(sobol) points that then get filtered to find all of them that are within the hull
    param_bounds = dict(np.load(r"Y:\Group Members\Valentin Aubry\01_Data\Parameters\Hull\parameter_bounds.npz"))
    
    # Define function to get an ndarray with the shape of the hull saved (find_simplex returns -1, which is outside of the hull shape)
    # This function thus tests if s point is within the convex hull shape(to find valid parameter iterations)
    def in_hull(p):
        # Needs delaunay to be a scipy.spacial.Delaunay object
        return delaunay.find_simplex(p) >= 0
    
    
    # Generate stratified parameter samples
    # Sobol samples are a way to generate "random" points that fill out a space(here: bounding box) very evenly
    n_parameter_samples = 256  # Amount of samples that are filtered for being inside the hull and kept
    n_sobol = 2048*2*2         # Amount of samples that are generated, to then be filtered for being inside the hull
    # We are oversampling by quite a bit, because later we are not just choosing points that are inside the hull, but we're also
    # trying to pick points as far apart as possible from eachother and with this the program has more options to choose from
    
    # Generates n_sobol samples in [0,1]^3 filling out the space evenly, d=3 means for 3 parameters, scramble=True improves distribution poperties
    sampler = qmc.Sobol(d=3, scramble=True)
    unit_samples = sampler.random(n=n_sobol)
    
    # Scale Sobol samples from [0,1]³ to your parameter bounds (Bounding box of the selected convex hull)
    # l_bounds are the minimum parameter values and u_bounds are the maxima(max and min from the bounding box of the convex hull)
    scaled_samples = qmc.scale(
        unit_samples,
        l_bounds=[
            param_bounds["bright_pct"][0],
            param_bounds["contrast_thresh"][0],
            param_bounds["percentile_val"][0],
        ],
        u_bounds=[
            param_bounds["bright_pct"][1],
            param_bounds["contrast_thresh"][1],
            param_bounds["percentile_val"][1],
        ],
    )
    
    # Filter all points inside the Delaunay hull
    inside_mask = in_hull(scaled_samples) # Using in_hull to find all points inside the convex hull
    valid_points = scaled_samples[inside_mask] # Array of all the valid points inside the hull
    # Using farthest_point_sampling to choose points inside the hull that are maximally far apart from eachother
    # Returns array (256,3) of final parameter iterations
    valid_param_samples = farthest_point_sampling(valid_points, n_samples=n_parameter_samples)
    # To be able to calculate the probability of each foci being detected sum of all parameter combinations
    total_iterations = len(valid_param_samples)

    # Reporting on the generated parameter iterations
    print(f" Generated {len(valid_param_samples)} valid parameter combinations")
    print(f"  Bright %: {valid_param_samples[:,0].min():.1f}-{valid_param_samples[:,0].max():.1f}")
    print(f"  Contrast: {valid_param_samples[:,1].min():.2f}-{valid_param_samples[:,1].max():.2f}")
    print(f"  Percentile: {valid_param_samples[:,2].min():.1f}-{valid_param_samples[:,2].max():.1f}")

    
    # CREATE LIST TO ACCUMULATE ALL FOCI DATA FOR ALL IMAGES
    all_foci_data = []
    
    # List to add all Data pertaining the Nuclei
    all_nuclei_data = []
     
    
    #High Throughput Data Loading Loop
    for i in range (len(DAPI_data)):
    #for i in range (1,2):
        #Loading the picture on i position from each channel
        Cy5_pic = imageio.imread(Cy5_data[i])
        DAPI_pic = imageio.imread(DAPI_data[i])
        DAPI_mask_pic = np.load(DAPI_mask_data[i], allow_pickle=True)
        FITC_pic = imageio.imread(FITC_data[i])
        TRITC_pic = imageio.imread(TRITC_data[i])
    
        # Extract the name of the picture that will be analyzed to later be able to save it accordingly
        base_name = os.path.basename(DAPI_data[i]).removesuffix('DAPI SEM.tif')
    
        # Extract the well number
        # searching the base name for the string --W and then all the numbers 0-9 with \d and the + to look for one or more of the numbers 0-9 with he () it saves them
        well_match = re.search(r'--W(\d+)', base_name)
        # Extracting just the numbers from the previously saved snippet(well_match)
        # group(1) gets just the numbers, group(0) would be with the --W as a prefix. If no numbers are found print of Error
        well_number = well_match.group(1) if well_match else None 
    
        # Extract the Posiiton number --> same principle like extraction of well number extraction
        position_match = re.search(r'--P(\d+)', base_name)
        position_number = position_match.group(1) if well_match else None

        #Warning message in case the well number or position number wasn't able to be extracted
        if well_number is None or position_number is None:
            print(f"Warning: missing Well/Position for file {base_name} (well={well_number}, pos={position_number})")
    
        # Converting the TRITC picture to float to make sure that all the pixel values are compatible with eachother
        TRITC_pic = img_as_float(TRITC_pic)
    
        # Extract the actual mask array
        masks = DAPI_mask_pic  # This contains the segmented regions
    
        
        # ------------------- PARALLEL NUCLEUS PROCESSING ------------------- #
        # Prepare list of nuclei (skip label 0 = background)
        cell_numbers = np.unique(masks)[1:] # This also tells the program where to start with the foci search(that's also why the background was dropped)
        #cell_numbers = [np.unique(masks)[42]]
        
        # Build argument tuples for each nucleus to send to workers
        # This will be pickled(convert data into a byte stream) to send them to the workers that are generated below
        tasks = [
            (cellnum, masks, TRITC_pic, FITC_pic, valid_param_samples, total_iterations, well_number, position_number)
            for cellnum in cell_numbers
        ]
        
        # Parallel processing 
        # This second main script check is just a safegard before all the workers for the paralellization are spawned(so they don't spawn themselves)
        # CRITICAL for Windows! On Windows ProcessPoolExecutor spawns fresh python interpreters that import the main module and they might re-run the whole thing otherwise
        if __name__ == '__main__':  
            # Spawns 8 workers max, leaves one CPU free(cpu_count()-1) and if only one core is available it spawns one worker
            max_workers = min(8, os.cpu_count() - 1) if os.cpu_count() > 1 else 1
            print(f"🚀 Using {max_workers} parallel workers for picture {(i+1)}/{len(DAPI_data)}")
            
            from tqdm import tqdm
            # Actual worker spawning by ProcessPoolExecutor
            with ProcessPoolExecutor(max_workers=max_workers) as executor:
                # executor.submit(process_single_nucleus, task --> this schedules each nucleus(process_single_nucleus function) as an independent job
                # futures maps the tasks so that with a future error the task that caused it ca nbe identified
                futures = {executor.submit(process_single_nucleus, task): task for task in tasks}
                # tdqm displays a progress bar, while as_completed(futures) the futures yields as they are completed
                for future in tqdm(as_completed(futures), total=len(futures), desc="Processing nuclei"):
                    try:
                        # Returns the lisits below that are then added onto the global lists where all the parameters are saved
                        foci_data_list, nuclei_data_list = future.result()
                        all_foci_data.extend(foci_data_list)
                        all_nuclei_data.extend(nuclei_data_list)

                    except Exception as e:
                        # Catching errors and printing where exactly it went wrong
                        task = futures[future]
                        print(f"❌ Error on nucleus {task[0]}: {e}")
 
    # Converting all the data that was previously saved in dictionnaries to dataframes to then save them
    # Here all the data that pertains nuclei is converted and then saved
    if all_nuclei_data:
        all_nuclei_df = pd.DataFrame(all_nuclei_data)
        all_nuclei_df = all_nuclei_df.sort_values(['Well', 'Position', 'cell_num']).reset_index(drop=True) #Sorts everything by Well, then Position and then by Cell number
        save_dataframe_to_csv(all_nuclei_df, folder_path, '3_param_iteration', 'nuclei')
    # Here all the data that pertains foci is converted and then saved
    if all_foci_data:
        all_foci_df = pd.DataFrame(all_foci_data)
        all_foci_df = all_foci_df.sort_values(['Well', 'Position', 'cell_num']).reset_index(drop=True) #Sorts everything by Well, then Position and then by Cell number
        save_dataframe_to_csv(all_foci_df, folder_path, '3_param_iteration', 'foci')
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    # Convert to minutes and seconds for easier reading
    minutes = int(elapsed_time // 60)
    seconds = elapsed_time % 60
    
    print(f"Total runtime: {minutes} minutes and {seconds:.2f} seconds")
    
    '''
    One Nucleus Runtime:
    Before optimization:
    Total runtime: 2 minutes and 17.56 seconds
    After middle part optimization:
    Total runtime: 0 minutes and 4.14 seconds


    734 Pictures:
    Total runtime: 6 hours 14 minutes and 40.45 seconds
    540 Pictures:
    Total runtime: 5 hours 19 minutes and 14.09 seconds
    '''

 Generated 256 valid parameter combinations
  Bright %: 51.0-85.4
  Contrast: 2.00-3.81
  Percentile: 3.9-91.9
🚀 Using 8 parallel workers for picture 1/540


Processing nuclei: 100%|███████████████████████████████████████████████████████████████| 33/33 [00:17<00:00,  1.87it/s]


🚀 Using 8 parallel workers for picture 2/540


Processing nuclei: 100%|███████████████████████████████████████████████████████████████| 34/34 [00:18<00:00,  1.86it/s]


🚀 Using 8 parallel workers for picture 3/540


...

🚀 Using 8 parallel workers for picture 539/540


Processing nuclei: 100%|███████████████████████████████████████████████████████████████| 87/87 [00:40<00:00,  2.14it/s]


🚀 Using 8 parallel workers for picture 540/540


Processing nuclei: 100%|█████████████████████████████████████████████████████████████| 113/113 [00:52<00:00,  2.14it/s]


Total runtime: 319 minutes and 14.09 seconds
